# Câu 1 – Xử lý Histogram ảnh số

**Bức tranh tổng quát:** một ảnh số chỉ là **một bảng số** — ảnh xám kích thước H×W là một
bảng có H dòng, W cột, mỗi ô chứa một số nguyên từ 0 (đen) đến 255 (trắng).

**Histogram** là một bảng đếm khác (256 ô, ứng với 256 mức sáng 0–255), cho biết
mỗi mức sáng xuất hiện bao nhiêu lần trong ảnh.

**4 bước của bài, mỗi bước chỉ làm 1 việc:**
1. Ảnh màu → ảnh xám
2. Ảnh xám → Histogram H1
3. Cân bằng histogram → ảnh mới + H2
4. Thu hẹp về [a, b] → ảnh mới + H3

**Mẹo nhớ — khuôn xử lý ảnh dùng lại 3 lần (bước 1, 3, 4):**
```
tạo ảnh/bảng kết quả rỗng, cùng kích thước
for y in range(H):          # duyệt từng dòng
    for x in range(W):      # duyệt từng cột
        # lấy giá trị cũ tại (y, x), áp 1 công thức toán, ghi giá trị mới vào (y, x)
```
Học khuôn này 1 lần là tự viết lại được toàn bộ bài.


## Bước 0: Tải ảnh lên (Google Colab)

`files.upload()` mở hộp thoại chọn file ngay trong Colab. Hàm này trả về một
**dictionary** dạng `{"ten_anh.jpg": <dữ liệu nhị phân của file>}`.

Ta chỉ cần lấy *tên file* để mở ảnh ở bước sau, nên:
- `uploaded.keys()` → lấy ra danh sách các khóa (chính là tên file) trong dictionary
- `list(...)` → đổi thành list để có thể lấy theo chỉ số
- `[0]` → lấy phần tử đầu tiên (vì chỉ upload 1 ảnh)


In [ ]:
from google.colab import files

uploaded = files.upload()                # mở hộp thoại chọn ảnh, trả về dictionary {ten_file: du_lieu}
IMAGE_PATH = list(uploaded.keys())[0]    # lấy tên file ảnh vừa tải lên (phần tử đầu tiên trong dictionary)


## Bước 1: Ảnh màu → Ảnh xám

**Công thức:**
$$xam = 0.299 \times R + 0.587 \times G + 0.114 \times B$$

Đây là công thức chuẩn (ITU-R BT.601). Hệ số của G (xanh lá) lớn nhất vì mắt người
nhạy với màu xanh lá hơn đỏ và xanh dương — cùng độ sáng vật lý nhưng mắt thấy
phần xanh lá "sáng" hơn, nên nó được tính trọng số cao hơn khi gộp 3 kênh màu
thành 1 mức xám.


In [ ]:
import numpy as np                  # thư viện xử lý mảng số, dùng để biểu diễn ảnh dưới dạng bảng số
import matplotlib.pyplot as plt     # thư viện vẽ hình, dùng để hiển thị ảnh và biểu đồ
from PIL import Image               # thư viện đọc/ghi file ảnh (jpg, png, ...)

# Image.open(...) đọc file ảnh từ đường dẫn IMAGE_PATH
# .convert("RGB") đảm bảo ảnh luôn có đúng 3 kênh màu Đỏ-Xanh lá-Xanh dương
#   (phòng trường hợp ảnh gốc là ảnh xám sẵn hoặc có thêm kênh trong suốt alpha)
# np.array(...) đổi ảnh đó thành một mảng số 3 chiều: img[y, x, c]
#   với c = 0 là kênh R, c = 1 là kênh G, c = 2 là kênh B
img = np.array(Image.open(IMAGE_PATH).convert("RGB"))

# img.shape trả về (số_dòng, số_cột, số_kênh_màu)
H, W = img.shape[0], img.shape[1]     # H = chiều cao (số dòng), W = chiều rộng (số cột)

# Tạo bảng "gray" rỗng, kích thước H dòng x W cột, mọi ô ban đầu = 0
# dtype=np.uint8 nghĩa là mỗi ô chỉ chứa số nguyên từ 0 đến 255 (đúng kiểu của ảnh xám)
gray = np.zeros((H, W), dtype=np.uint8)

# Duyệt qua TỪNG pixel của ảnh: y chạy hết các dòng, với mỗi dòng x chạy hết các cột
for y in range(H):
    for x in range(W):
        # Lấy 3 giá trị màu tại đúng pixel (y, x)
        R, G, B = img[y, x, 0], img[y, x, 1], img[y, x, 2]

        # Áp dụng công thức chuyển sang mức xám, round() để được số nguyên,
        # rồi ghi kết quả vào đúng vị trí (y, x) của bảng "gray"
        gray[y, x] = round(0.299*R + 0.587*G + 0.114*B)

# Hiển thị ảnh xám vừa tạo
plt.imshow(gray, cmap="gray")   # cmap="gray" để hiển thị đúng thang màu đen-trắng
plt.title("Ảnh xám")
plt.axis("off")                 # tắt hiển thị trục số (vì đây là ảnh, không phải biểu đồ)
plt.show()


## Bước 2: Tính Histogram H1

**Công thức:** với mỗi pixel, `H[mức_sáng_của_pixel] += 1`

Nói cách khác: tạo 256 "ô đếm" (ứng với 256 mức sáng có thể có), rồi duyệt qua
toàn bộ ảnh — pixel nào có mức sáng `k` thì ô đếm thứ `k` được cộng thêm 1.
Sau khi duyệt hết ảnh, `H1[k]` chính là số lượng pixel có mức sáng bằng `k`.


In [ ]:
H1 = [0] * 256   # tạo list 256 số 0: H1[0], H1[1], ..., H1[255] — mỗi ô ứng với 1 mức sáng

# Duyệt qua từng pixel của ảnh xám
for y in range(H):
    for x in range(W):
        muc_sang = gray[y, x]   # mức sáng (0-255) tại pixel (y, x)
        H1[muc_sang] += 1       # dùng mức sáng đó làm CHỈ SỐ để tăng ô đếm tương ứng lên 1

# Vẽ histogram: trục ngang là mức sáng (0-255), trục dọc là số pixel có mức sáng đó
plt.bar(range(256), H1)
plt.title("H1 - Histogram ảnh gốc")
plt.xlabel("Mức sáng (0-255)")
plt.ylabel("Số lượng pixel")
plt.show()


## Bước 3: Cân bằng Histogram (Histogram Equalization)

**Mục đích:** nếu histogram gốc dồn cục ở một vùng hẹp (ảnh quá tối/quá sáng/ít
tương phản), cân bằng histogram sẽ "kéo dãn" các mức sáng ra để chúng phủ đều
khắp khoảng 0–255, giúp ảnh rõ và có độ tương phản tốt hơn.

**3 công thức theo đúng 3 bước nhỏ:**
```
CDF[k]  = H1[0] + H1[1] + ... + H1[k]      (cộng dồn histogram — gọi là "histogram tích lũy")
moi[k]  = round( CDF[k] / (H*W) * 255 )    (đổi tỉ lệ tích lũy thành mức sáng mới 0-255)
anh_moi[y,x] = moi[ gray[y,x] ]            (tra bảng "moi" để đổi từng pixel sang giá trị mới)
```

Vì sao công thức này lại "kéo dãn" histogram? Nếu một vùng mức sáng có rất nhiều
pixel (cột histogram cao), thì `CDF` sẽ tăng nhanh qua vùng đó, kéo theo `moi[k]`
cũng tăng nhanh — nghĩa là vùng đông pixel sẽ được "trải ra" thành một dải rộng hơn
trên thang 0–255. Ngược lại vùng ít pixel sẽ bị nén lại.


In [ ]:
# --- 3.1. Cộng dồn H1 thành CDF (histogram tích lũy) ---
CDF = [0] * 256   # tạo list 256 số 0 để chứa giá trị tích lũy
tong = 0          # biến "tong" đóng vai trò bộ đếm cộng dồn, bắt đầu từ 0

for k in range(256):
    tong += H1[k]     # cộng thêm số pixel ở mức sáng k vào tổng đang tích lũy
    CDF[k] = tong      # gán tổng hiện tại vào CDF[k]
    # => CDF[k] = H1[0] + H1[1] + ... + H1[k]  (tổng số pixel có mức sáng TỪ 0 ĐẾN k)

# --- 3.2. Tạo bảng tra cứu: đổi mỗi mức sáng cũ k thành mức sáng mới moi[k] ---
moi = [0] * 256
for k in range(256):
    # CDF[k] / (H * W)  -> tỉ lệ phần trăm pixel có mức sáng <= k (một số từ 0 đến 1)
    # nhân với 255      -> trải tỉ lệ đó ra lại thành thang đo 0-255
    moi[k] = round(CDF[k] / (H * W) * 255)

# --- 3.3. Áp dụng bảng tra cứu "moi" cho từng pixel của ảnh xám ---
eq_img = np.zeros((H, W), dtype=np.uint8)   # tạo ảnh kết quả rỗng, cùng kích thước ảnh gốc

for y in range(H):
    for x in range(W):
        muc_sang_cu = gray[y, x]      # mức sáng cũ tại pixel (y, x)
        eq_img[y, x] = moi[muc_sang_cu]   # tra bảng "moi" để lấy mức sáng mới, ghi vào ảnh kết quả

# Hiển thị ảnh sau khi cân bằng
plt.imshow(eq_img, cmap="gray")
plt.title("Ảnh sau cân bằng")
plt.axis("off")
plt.show()


In [ ]:
# Tính H2 = histogram của ảnh SAU khi cân bằng
# Cách làm GIỐNG HỆT Bước 2, chỉ đổi đầu vào từ "gray" thành "eq_img"
H2 = [0] * 256
for y in range(H):
    for x in range(W):
        H2[eq_img[y, x]] += 1

plt.bar(range(256), H2, color="orange")
plt.title("H2 - Histogram sau cân bằng")
plt.xlabel("Mức sáng (0-255)")
plt.ylabel("Số lượng pixel")
plt.show()


## Bước 4: Thu hẹp Histogram về khoảng [a, b]

**Mục đích:** ngược lại với cân bằng — ép toàn bộ mức sáng (đang trải khắp 0–255)
phải nén lại, chỉ còn nằm trong một khoảng hẹp hơn, ví dụ [a, b] = [30, 120].

**Công thức (phép biến đổi tuyến tính, giống `y = a + độ_dốc × x`):**
$$k_{moi} = round\left(a + \dfrac{b-a}{255} \times k\right)$$

- Khi `k = 0`  → kết quả = `a` (giá trị nhỏ nhất của ảnh sau biến đổi)
- Khi `k = 255` → kết quả = `a + (b - a) = b` (giá trị lớn nhất)
- Các giá trị `k` ở giữa sẽ rơi vào đúng vị trí tỉ lệ tương ứng giữa `a` và `b`

Đầu vào của bước này là `eq_img` (ảnh **sau khi đã cân bằng** ở Bước 3), vì bài
yêu cầu làm nối tiếp theo đúng thứ tự: ảnh gốc → cân bằng → thu hẹp.


In [ ]:
a, b = 30, 120   # khoảng mức sáng mong muốn sau khi thu hẹp (có thể đổi 2 số này)

shrunk_img = np.zeros((H, W), dtype=np.uint8)   # tạo ảnh kết quả rỗng

for y in range(H):
    for x in range(W):
        k = eq_img[y, x]     # lấy mức sáng tại pixel (y, x) của ảnh ĐÃ CÂN BẰNG
        # áp công thức biến đổi tuyến tính để nén mức sáng k (0-255) về khoảng [a, b]
        shrunk_img[y, x] = round(a + (b - a) / 255 * k)

# Hiển thị ảnh sau khi thu hẹp
plt.imshow(shrunk_img, cmap="gray")
plt.title(f"Ảnh sau thu hẹp [{a},{b}]")
plt.axis("off")
plt.show()


In [ ]:
# Tính H3 = histogram của ảnh SAU khi thu hẹp
# Cách làm vẫn GIỐNG HỆT Bước 2, chỉ đổi đầu vào thành "shrunk_img"
H3 = [0] * 256
for y in range(H):
    for x in range(W):
        H3[shrunk_img[y, x]] += 1

plt.bar(range(256), H3, color="green")
plt.title("H3 - Histogram sau thu hẹp")
plt.xlabel("Mức sáng (0-255)")
plt.ylabel("Số lượng pixel")
plt.show()


## Bước 5: Hiển thị tổng hợp & lưu kết quả

Ghép cả 3 ảnh và 3 histogram vào một lưới duy nhất (3 dòng × 2 cột) để so sánh
trực quan: dòng 1 là ảnh gốc, dòng 2 là sau cân bằng, dòng 3 là sau thu hẹp.


In [ ]:
# plt.subplots(3, 2, ...) tạo một lưới 3 dòng, 2 cột để vẽ 6 ô cùng lúc
# "axes" là một bảng các ô vẽ: axes[dòng, cột]
fig, axes = plt.subplots(3, 2, figsize=(12, 12))

# --- Dòng 0: ảnh xám gốc (cột 0) và histogram H1 (cột 1) ---
axes[0,0].imshow(gray, cmap="gray")          # vẽ ảnh xám gốc vào ô (dòng 0, cột 0)
axes[0,0].set_title("Ảnh xám gốc")
axes[0,0].axis("off")                         # tắt trục số vì đây là ảnh
axes[0,1].bar(range(256), H1)                 # vẽ histogram H1 vào ô (dòng 0, cột 1)
axes[0,1].set_title("H1")

# --- Dòng 1: ảnh sau cân bằng (cột 0) và histogram H2 (cột 1) ---
axes[1,0].imshow(eq_img, cmap="gray")
axes[1,0].set_title("Sau cân bằng")
axes[1,0].axis("off")
axes[1,1].bar(range(256), H2, color="orange")
axes[1,1].set_title("H2")

# --- Dòng 2: ảnh sau thu hẹp (cột 0) và histogram H3 (cột 1) ---
axes[2,0].imshow(shrunk_img, cmap="gray")
axes[2,0].set_title(f"Sau thu hẹp [{a},{b}]")
axes[2,0].axis("off")
axes[2,1].bar(range(256), H3, color="green")
axes[2,1].set_title("H3")

plt.tight_layout()                              # tự động chỉnh khoảng cách giữa các ô cho gọn
plt.savefig("cau1_ket_qua.png", dpi=150)        # lưu cả lưới hình thành 1 file ảnh PNG
plt.show()                                       # hiển thị lưới hình lên màn hình
